# `main_96x96` audit against the Diffusion Policy paper

This notebook reviews whether the `main_96x96` implementation is a faithful implementation of **Diffusion Policy: Visuomotor Policy Learning via Action Diffusion** and records likely causes of the observed on-arm jitter on the SO-ARM101.

Scope:
- compare the code and saved run config against the paper's described CNN-based policy;
- identify implementation deviations and control-loop issues that can plausibly cause jitter;
- document stale or misleading project docs that make debugging harder.

Bottom line:
- this is **not** a paper-faithful reproduction of the vision stack;
- the largest control-side omission is **no warm-start between replans**;
- the current inference loop also adds a software clip against the **last commanded** target, which can create visible twitching if the arm lags the command stream;
- several repo docs describe a different run than the saved `main_96x96` config.

In [1]:
from pathlib import Path
import json
from omegaconf import OmegaConf

RUN_DIR = Path("../../runs/main_96x96/20260528_185334")
cfg = OmegaConf.load(RUN_DIR / "config.yaml")

print("Saved run config summary:\n")
for key in [
    "model.obs_horizon",
    "model.pred_horizon",
    "model.exec_horizon",
    "encoder.pretrained",
    "encoder.freeze_bn",
    "diffusion.sampler",
    "diffusion.num_inference_steps",
    "diffusion.clip_sample",
    "training.num_epochs",
    "infer.num_ddim_steps",
    "infer.max_relative_target",
]:
    print(f"{key}: {OmegaConf.select(cfg, key)}")

print("\nAction normalizer ranges:\n")
norm = json.loads((RUN_DIR / "action_normalizer.json").read_text())
for i, (lo, hi) in enumerate(zip(norm["mins"], norm["maxs"])):
    print(f"joint {i}: [{lo:8.3f}, {hi:8.3f}]  span={hi-lo:8.3f}")


Saved run config summary:

model.obs_horizon: 2
model.pred_horizon: 16
model.exec_horizon: 4
encoder.pretrained: True
encoder.freeze_bn: False
diffusion.sampler: ddpm
diffusion.num_inference_steps: 100
diffusion.clip_sample: True
training.num_epochs: 300
infer.num_ddim_steps: 10
infer.max_relative_target: 10.0

Action normalizer ranges:

joint 0: [ -73.538,   50.154]  span= 123.692
joint 1: [-103.560,   64.000]  span= 167.560
joint 2: [ -71.824,   96.879]  span= 168.703
joint 3: [  33.275,  101.143]  span=  67.868
joint 4: [ -78.989,   82.505]  span= 161.495
joint 5: [   0.492,   43.232]  span=  42.740


## Finding 1: the visual encoder does not match the paper

The paper's vision encoder section describes three choices that matter here:
- ResNet-18 **without pretraining**;
- replace **global average pooling** with **spatial softmax**;
- replace **BatchNorm** with **GroupNorm** for stability with EMA.

The current implementation does something else:
- `diffusion_policy_soarm/models/encoders.py:46-50` loads a standard torchvision ResNet-18 and keeps the default average-pool path;
- `diffusion_policy_soarm/models/encoders.py:46` enables ImageNet weights when `pretrained=true`;
- `diffusion_policy_soarm/models/encoders.py:64-67` only optionally freezes BatchNorm, it does not replace it;
- the saved `main_96x96` run config keeps `encoder.pretrained: true` and `encoder.freeze_bn: false` in `runs/main_96x96/20260528_185334/config.yaml:17-20`.

Why this matters:
- average pooling throws away spatial detail that is important for manipulation;
- keeping BatchNorm instead of GroupNorm is a direct mismatch with the paper's stability recommendation;
- ImageNet pretraining is not automatically wrong, but it is another departure from the setup you asked to reproduce.

Verdict: this is the clearest paper-faithfulness issue in `main_96x96`.

## Finding 2: inference does not warm-start the next action chunk

The paper's closed-loop action-sequence discussion explicitly says receding-horizon control can be improved by **warm-starting the next inference with the previous predicted action sequence**.

That is not implemented here:
- `diffusion_policy_soarm/models/diffusion.py:315` starts every DDIM sample from fresh Gaussian noise with `x = torch.randn(...)`;
- `diffusion_policy_soarm/infer.py:240-241` calls `model.predict_actions(batch)` once per replan;
- `diffusion_policy_soarm/infer.py:251-278` executes the first `exec_horizon` actions and then discards the rest.

There is no path that reuses the unexecuted suffix of the previous chunk, and no path that seeds the next denoising trajectory from the previous plan.

Why this matters for the jitter you saw:
- every replan can land in a slightly different local mode even for nearly identical observations;
- with `exec_horizon=4` in the saved run config (`runs/main_96x96/20260528_185334/config.yaml:13-16`), replanning happens every 133 ms at 30 Hz, so any chunk boundary discontinuity is seen often;
- this directly weakens the temporal consistency benefit that the paper claims for chunked diffusion policies.

Verdict: this is likely one of the main reasons the arm appears twitchy even when latency is acceptable.

## Finding 3: the software safety clip can create command/measurement mismatch

The paper does not include the repo's extra software-side target limiter. The current inference loop adds one:
- `diffusion_policy_soarm/infer.py:261-275` clips each requested joint delta against `prev_action`, which is the **last commanded** position, not the latest measured position;
- the default config enables it through `infer.max_relative_target` in `diffusion_policy_soarm/configs/base.yaml:138-144`, and the saved run widens it to `10.0` deg in `runs/main_96x96/20260528_185334/config.yaml:97-100`.

Why this can cause visible jitter:
- if the servo has not yet reached the previous command, the next clip is computed from an internal target that does not match the physical arm state;
- if the policy wants to reverse direction at the next chunk, the limiter can produce alternating under-shoot / catch-up behavior;
- this is especially noticeable when combined with fresh-noise replanning and absolute joint targets.

This clip is reasonable as a safety layer, but it changes the deployed controller enough that you should not treat the observed motion as pure model behavior.

## Finding 4: `clip_sample=True` makes out-of-distribution starts fail hard

Both samplers clamp the reconstructed clean action estimate every denoising step:
- `diffusion_policy_soarm/models/diffusion.py:240-241` in DDPM;
- `diffusion_policy_soarm/models/diffusion.py:328-329` in DDIM.

That is consistent with the repo's min-max normalization, but it has an important deployment consequence: if the current observation is far from the training distribution, the denoiser can saturate to `-1` or `+1`, which maps exactly to the joint-wise min/max in `action_normalizer.json`.

The saved action ranges are quite wide, for example:
- joint 1: `[-103.56, 64.00]`
- joint 2: `[-71.82, 96.88]`

If the arm is started from a pose not represented in demonstrations, the first few inferred chunks can rail to those bounds and look like jitter or frozen twitching rather than purposeful motion.

Verdict: this does not prove the implementation is wrong, but it is a strong explanation for the specific 'jitters in place' failure mode you described.

## Finding 5: the repo docs around `main_96x96` are stale and internally inconsistent

The saved run config says:
- `exec_horizon: 4` in `runs/main_96x96/20260528_185334/config.yaml:13-16`;
- `training.num_epochs: 300` in `runs/main_96x96/20260528_185334/config.yaml:46-60`;
- inference is forced to DDIM from `infer.py:308-314` and the run uses `infer.num_ddim_steps: 10` in `runs/main_96x96/20260528_185334/config.yaml:97-100`.

But the docs say different things:
- `README.md:68-80` describes executing the first 8 actions;
- `README.md:116` says the run was 150 epochs;
- `diffusion_policy_soarm/docs/how_it_works.md:105-113` says `T_a=8`;
- `diffusion_policy_soarm/docs/writeup.md:60-62` says inference is DDPM with 100 steps and `T_a=8`.

This matters because it is easy to debug the wrong thing if you trust the prose instead of the saved run config.

Verdict: use the run config and code, not the README/notebook prose, as the ground truth for `main_96x96`.

## Overall verdict

Is `main_96x96` correct as an implementation of the paper?

Not strictly. The core diffusion loss and FiLM-conditioned 1-D CNN denoiser are aligned with the paper's high-level method, but the deployed `main_96x96` setup differs in several important places:
- the visual encoder is not paper-faithful;
- the receding-horizon inference loop omits warm-starting;
- the on-arm controller adds a non-paper safety limiter that can itself create twitchy behavior;
- the docs do not accurately describe the saved run.

Most likely jitter contributors, in order:
1. no warm-start between replans;
2. software clipping against last commanded state instead of measured state;
3. out-of-distribution start pose causing `clip_sample` saturation;
4. reduced spatial precision from average-pool ResNet features.

If the goal is to follow the article more closely, the first implementation changes I would make are:
1. replace the current visual encoder with a ResNet-18 that uses spatial softmax and GroupNorm, and remove ImageNet pretraining for the paper-faithful run;
2. add warm-starting so the next DDIM sample is initialized from the previous plan's suffix instead of fresh Gaussian noise;
3. make the safety limiter optional during diagnosis, or at least clip against the latest measured pose rather than `prev_action`;
4. update the repo docs so they describe the actual saved run.